Week 3

In [ ]:

import itertools
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from tensorflow.keras.applications import MobileNetV2, ResNet50

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ── 9.1  Configuration — kept identical to Week 2 for fair comparison
IMG_SIZE   = (224, 224)
SEED       = 42
AUTOTUNE   = tf.data.AUTOTUNE

tf.random.set_seed(SEED)
np.random.seed(SEED)

# ImageNet preprocessing helper (used by both MobileNetV2 & ResNet50)
# We normalise to [-1, 1] for MobileNetV2 and use keras preprocess_input for ResNet50.
# A unified pipeline is built per-model inside build_transfer_model().

print("Week 3 configuration ready ✅")

In [ ]:
# ── 9.2  tf.data pipelines with model-specific preprocessing

def load_image(path, label):
    """Read, decode and resize; normalisation done inside the model."""
    raw   = tf.io.read_file(path)
    image = tf.image.decode_jpeg(raw, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)   # keep in [0, 255] — backbone handles scaling
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def augment_image(image, label):
    """Richer augmentation for transfer learning (more robust features)."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.3)
    image = tf.image.random_contrast(image, lower=0.7, upper=1.3)
    image = tf.image.random_saturation(image, lower=0.8, upper=1.2)
    image = tf.image.random_hue(image, max_delta=0.05)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

def build_ds(paths, labels, batch_size=32, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if augment:
        ds = ds.map(augment_image, num_parallel_calls=AUTOTUNE)
    return ds.batch(batch_size).prefetch(AUTOTUNE)

print("Dataset builder ready ✅")

In [ ]:
# ── 9.3  Model factory — builds a transfer-learning model for any backbone

def build_transfer_model(
    backbone_name: str = 'MobileNetV2',
    num_classes:   int = NUM_CLASSES,
    dropout_rate:  float = 0.4,
    dense_units:   int = 256,
    learning_rate: float = 1e-3,
    freeze_base:   bool = True,
):
    """
    Build a transfer-learning model.

    Parameters
    ----------
    backbone_name : 'MobileNetV2' | 'ResNet50'
    freeze_base   : True  → Phase 1 (head-only training)
                    False → Phase 2 (fine-tuning top layers)
    """
    # ── Select backbone
    if backbone_name == 'MobileNetV2':
        preprocess_fn = tf.keras.applications.mobilenet_v2.preprocess_input   # [0,255]→[-1,1]
        base = MobileNetV2(
            input_shape=(*IMG_SIZE, 3),
            include_top=False,
            weights='imagenet'
        )
        fine_tune_from = 100   # unfreeze layers after this index in Phase 2
    elif backbone_name == 'ResNet50':
        preprocess_fn = tf.keras.applications.resnet50.preprocess_input       # RGB mean subtraction
        base = ResNet50(
            input_shape=(*IMG_SIZE, 3),
            include_top=False,
            weights='imagenet'
        )
        fine_tune_from = 140   # unfreeze last ~15 residual blocks
    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    # ── Freeze / unfreeze
    base.trainable = not freeze_base
    if not freeze_base:
        # Fine-tune only the top layers; keep early layers frozen
        for layer in base.layers[:fine_tune_from]:
            layer.trainable = False
        for layer in base.layers[fine_tune_from:]:
            layer.trainable = True

    n_trainable = sum(1 for l in base.layers if l.trainable)
    print(f"[{backbone_name}] Trainable backbone layers: {n_trainable} / {len(base.layers)}")

    # ── Build functional model
    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='image_input')
    x = layers.Lambda(preprocess_fn, name='backbone_preprocess')(inputs)
    x = base(x, training=False)          # training=False keeps BN in inference mode during Phase 1
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_head')(x)
    x = layers.Dense(dense_units, activation='relu', name='fc1')(x)
    x = layers.Dropout(dropout_rate, name='drop1')(x)
    x = layers.Dense(dense_units // 2, activation='relu', name='fc2')(x)
    x = layers.Dropout(dropout_rate / 2, name='drop2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = keras.Model(inputs, outputs, name=f'PlantDisease_{backbone_name}')

    # ── Compile
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
        ]
    )
    return model, base, fine_tune_from

# Quick sanity check
_ , _base, _ft = build_transfer_model('MobileNetV2')
print(f"MobileNetV2 total backbone layers : {len(_base.layers)}")
del _, _base, _ft

Hyperparameter Grid Search (Phase 1 — Frozen Base)



In [ ]:
# ── 10.1  Hyperparameter grid
HP_GRID = {
    'backbone'       : ['MobileNetV2', 'ResNet50'],
    'learning_rate'  : [1e-2, 1e-3, 5e-4],
    'batch_size'     : [32, 64],
}

HP_SEARCH_EPOCHS = 5      # quick probe — enough to see trajectory
hp_results = []           # will store dicts of {backbone, lr, bs, val_acc, val_loss}

combos = list(itertools.product(
    HP_GRID['backbone'],
    HP_GRID['learning_rate'],
    HP_GRID['batch_size']
))
print(f"Total hyperparameter combinations: {len(combos)}")
for i, (bb, lr, bs) in enumerate(combos):
    print(f"  [{i+1:2d}] backbone={bb:12s}  lr={lr:.0e}  batch={bs}")

In [ ]:
# ── 10.2  Run the grid search
# ⏳ Each combo trains for HP_SEARCH_EPOCHS epochs — expect ~10-20 min total on GPU.

for run_idx, (backbone, lr, bs) in enumerate(combos):
    print(f"\n{'='*60}")
    print(f" Run {run_idx+1}/{len(combos)} | {backbone} | LR={lr:.0e} | BS={bs}")
    print(f"{'='*60}")

    # Build datasets for this batch size
    tr_ds = build_ds(train_paths, train_labels, batch_size=bs, shuffle=True,  augment=True)
    vl_ds = build_ds(val_paths,   val_labels,   batch_size=bs, shuffle=False, augment=False)

    # Build model (Phase 1: frozen base)
    model, _, _ = build_transfer_model(
        backbone_name=backbone,
        learning_rate=lr,
        freeze_base=True
    )

    t0 = time.time()
    hist = model.fit(
        tr_ds,
        validation_data=vl_ds,
        epochs=HP_SEARCH_EPOCHS,
        class_weight=class_weight_dict,
        verbose=1
    )
    elapsed = time.time() - t0

    best_val_acc  = max(hist.history['val_accuracy'])
    best_val_loss = min(hist.history['val_loss'])

    hp_results.append({
        'backbone'     : backbone,
        'learning_rate': lr,
        'batch_size'   : bs,
        'val_accuracy' : best_val_acc,
        'val_loss'     : best_val_loss,
        'time_s'       : elapsed
    })

    print(f" ✅ Best val_acc: {best_val_acc:.4f}  |  Best val_loss: {best_val_loss:.4f}  |  Time: {elapsed:.0f}s")
    del model   # free VRAM
    tf.keras.backend.clear_session()

print("\n🎉 Grid search complete!")

In [ ]:
# ── 10.3  Results table
hp_df = pd.DataFrame(hp_results).sort_values('val_accuracy', ascending=False).reset_index(drop=True)
hp_df['val_accuracy'] = hp_df['val_accuracy'].map('{:.4f}'.format)
hp_df['val_loss']     = hp_df['val_loss'].map('{:.4f}'.format)
hp_df['learning_rate']= hp_df['learning_rate'].map('{:.0e}'.format)
hp_df['time_s']       = hp_df['time_s'].map('{:.0f}s'.format)
print(hp_df.to_string(index=True))

hp_df.to_csv('hp_search_results.csv', index=False)
print("\nSaved: hp_search_results.csv")

In [ ]:
# ── 10.4  Visualise hyperparameter search results

results_raw = pd.DataFrame(hp_results)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, backbone in zip(axes, ['MobileNetV2', 'ResNet50']):
    sub = results_raw[results_raw['backbone'] == backbone]

    pivot = sub.pivot_table(
        index='batch_size',
        columns='learning_rate',
        values='val_accuracy'
    )

    import seaborn as sns
    sns.heatmap(
        pivot,
        annot=True, fmt='.4f',
        cmap='YlGn',
        vmin=results_raw['val_accuracy'].min(),
        vmax=results_raw['val_accuracy'].max(),
        ax=ax,
        linewidths=0.5
    )
    ax.set_title(f'{backbone} — Val Accuracy Heatmap', fontweight='bold', fontsize=12)
    ax.set_xlabel('Learning Rate', fontsize=10)
    ax.set_ylabel('Batch Size', fontsize=10)

plt.suptitle('Hyperparameter Search: Val Accuracy (5-epoch probe)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('hp_search_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: hp_search_heatmap.png")

In [ ]:
# ── 10.5  Extract best hyperparameters
best_row = pd.DataFrame(hp_results).sort_values('val_accuracy', ascending=False).iloc[0]

BEST_BACKBONE = best_row['backbone']
BEST_LR       = float(best_row['learning_rate'])
BEST_BS       = int(best_row['batch_size'])

print("\n🏆 Best hyperparameters found:")
print(f"   Backbone       : {BEST_BACKBONE}")
print(f"   Learning Rate  : {BEST_LR:.0e}")
print(f"   Batch Size     : {BEST_BS}")
print(f"   Val Accuracy   : {float(best_row['val_accuracy']):.4f}")

 Phase 1 — Full Head Training (Frozen Backbone)



In [ ]:
# ── 11.1  Build datasets with the best batch size
train_ds_p1 = build_ds(train_paths, train_labels, batch_size=BEST_BS, shuffle=True,  augment=True)
val_ds_p1   = build_ds(val_paths,   val_labels,   batch_size=BEST_BS, shuffle=False, augment=False)
test_ds_p1  = build_ds(test_paths,  test_labels,  batch_size=BEST_BS, shuffle=False, augment=False)

print(f"Train batches : {len(train_ds_p1)}")
print(f"Val   batches : {len(val_ds_p1)}")

In [ ]:
# ── 11.2  Build Phase 1 model (frozen backbone)
tl_model, tl_base, FINE_TUNE_FROM = build_transfer_model(
    backbone_name=BEST_BACKBONE,
    learning_rate=BEST_LR,
    freeze_base=True,
    dropout_rate=0.4,
    dense_units=256
)
tl_model.summary(line_length=100)

In [ ]:
# ── 11.3  Callbacks for Phase 1
P1_CKPT = 'tl_phase1_best.keras'

p1_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=P1_CKPT,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    callbacks.CSVLogger('tl_phase1_log.csv')
]

print("Phase 1 callbacks ready ✅")

In [ ]:
# ── 11.4  Train Phase 1
P1_EPOCHS = 20

print(f"Phase 1: Training classification head for up to {P1_EPOCHS} epochs...")
print(f"Backbone frozen — only {sum(1 for l in tl_model.layers if l.trainable)} layers are trainable.\n")

history_p1 = tl_model.fit(
    train_ds_p1,
    validation_data=val_ds_p1,
    epochs=P1_EPOCHS,
    callbacks=p1_callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

p1_best_val_acc = max(history_p1.history['val_accuracy'])
print(f"\nPhase 1 complete — best val accuracy: {p1_best_val_acc:.4f} ({p1_best_val_acc*100:.2f}%)")

 Phase 2 — Fine-Tuning Top Backbone Layers


In [ ]:
# ── 12.1  Load best Phase 1 checkpoint and unfreeze top backbone layers
tl_model = keras.models.load_model(P1_CKPT)

# Unfreeze the backbone sub-model inside the functional model
backbone_layer = next(l for l in tl_model.layers if isinstance(l, keras.Model))
backbone_layer.trainable = True

# Re-freeze early layers (keep low-level edge/colour detectors frozen)
for layer in backbone_layer.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

n_trainable = sum(1 for l in tl_model.layers if l.trainable)
print(f"Phase 2: Trainable layers = {n_trainable} / {len(tl_model.layers)} (model-level)")
print(f"         Backbone layers fine-tuned from index {FINE_TUNE_FROM} onward")

In [ ]:
# ── 12.2  Recompile with 10× lower learning rate
FINE_TUNE_LR = BEST_LR / 10.0

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
    ]
)
print(f"Model recompiled for fine-tuning | LR = {FINE_TUNE_LR:.0e}")

In [ ]:
# ── 12.3  Callbacks for Phase 2 (fine-tuning)
P2_CKPT = 'tl_phase2_best.keras'

p2_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=P2_CKPT,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,
        verbose=1
    ),
    callbacks.CSVLogger('tl_phase2_log.csv')
]

print("Phase 2 callbacks ready ✅")

In [ ]:
# ── 12.4  Fine-tune (Phase 2)
P2_EPOCHS = 30
TOTAL_EPOCHS = P1_EPOCHS + P2_EPOCHS   # for correct x-axis when plotting combined curves

print(f"Phase 2: Fine-tuning for up to {P2_EPOCHS} additional epochs...")

history_p2 = tl_model.fit(
    train_ds_p1,
    validation_data=val_ds_p1,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history_p1.history['loss']),   # continue epoch numbering
    callbacks=p2_callbacks,
    class_weight=class_weight_dict,
    verbose=1
)

p2_best_val_acc = max(history_p2.history['val_accuracy'])
print(f"\nPhase 2 complete — best val accuracy: {p2_best_val_acc:.4f} ({p2_best_val_acc*100:.2f}%)")

Training Curves & Evaluation

In [ ]:
# ── 13.1  Combined training curves (Phase 1 + Phase 2)

def combine_histories(h1, h2):
    """Concatenate two Keras history dicts into one."""
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history.get(k, [])
    return combined

all_history = combine_histories(history_p1, history_p2)
epochs_total = range(1, len(all_history['loss']) + 1)
phase1_len   = len(history_p1.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, title in zip(
    axes,
    [('loss', 'val_loss'), ('accuracy', 'val_accuracy')],
    ['Loss (Categorical Cross-Entropy)', 'Top-1 Accuracy']
):
    train_key, val_key = metric
    ax.plot(epochs_total, all_history[train_key], 'b-o', markersize=3, label=f'Train {train_key}')
    ax.plot(epochs_total, all_history[val_key],   'r-o', markersize=3, label=f'Val {val_key}')
    ax.axvline(phase1_len + 0.5, color='gray', linestyle='--', linewidth=1.5, label='Fine-tune starts')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.legend(fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle(
    f'Training Curves — {BEST_BACKBONE} Transfer Learning\n'
    f'(Phase 1: Frozen Base  |  Phase 2: Fine-Tuning)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('tl_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tl_training_curves.png")

In [ ]:
# ── 13.2  Test-set evaluation of the best fine-tuned model
from sklearn.metrics import classification_report, confusion_matrix

best_tl_model = keras.models.load_model(P2_CKPT)

tl_loss, tl_acc, tl_top5 = best_tl_model.evaluate(test_ds_p1, verbose=1)

print(f"\n{'='*50}")
print(f"  Transfer Learning Test Results ({BEST_BACKBONE})")
print(f"{'='*50}")
print(f"  Test Loss      : {tl_loss:.4f}")
print(f"  Test Accuracy  : {tl_acc:.4f}  ({tl_acc*100:.2f}%)")
print(f"  Top-5 Accuracy : {tl_top5:.4f}  ({tl_top5*100:.2f}%)")
print(f"{'='*50}")

if tl_acc >= 0.90:
    print("\n🎯 TARGET REACHED: Val accuracy ≥ 90% ✅")
else:
    print(f"\n⚠️  Still {(0.90 - tl_acc)*100:.1f}% below the 90% target — consider more epochs or data augmentation.")

In [ ]:
# ── 13.3  Per-class metrics
y_true_tl, y_pred_tl = [], []
for images, labels in test_ds_p1:
    preds = best_tl_model.predict(images, verbose=0)
    y_true_tl.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_tl.extend(np.argmax(preds, axis=1))

y_true_tl = np.array(y_true_tl)
y_pred_tl = np.array(y_pred_tl)

report_tl = classification_report(
    y_true_tl, y_pred_tl,
    target_names=classes,
    output_dict=True
)

# Show top 10 best and bottom 5 worst performing classes
class_f1 = {cls: report_tl[cls]['f1-score'] for cls in classes}
sorted_f1 = sorted(class_f1.items(), key=lambda x: x[1], reverse=True)

print("\n📈 Top-10 Best Performing Classes (F1):")
for cls, f1 in sorted_f1[:10]:
    print(f"   {cls[:50]:50s}  F1={f1:.3f}")

print("\n📉 Bottom-5 Worst Performing Classes (F1):")
for cls, f1 in sorted_f1[-5:]:
    print(f"   {cls[:50]:50s}  F1={f1:.3f}")

In [ ]:
# ── 13.4  Confusion Matrix (Transfer Learning model)
cm_tl = confusion_matrix(y_true_tl, y_pred_tl)

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm_tl, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_title(f'Confusion Matrix — {BEST_BACKBONE} (Test Set)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
tick_labels = [c.split('___')[-1].replace('_', ' ')[:20] for c in classes]
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=6)
ax.set_yticklabels(tick_labels, fontsize=6)
plt.tight_layout()
plt.savefig('tl_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tl_confusion_matrix.png")

In [ ]:
# ── 13.5  Head-to-head: Custom CNN (Week 2) vs Transfer Learning (Week 3)
# Load Week 2 test accuracy from saved model (if available)
import os

W2_CKPT = 'plant_disease_cnn_final.keras'
if os.path.exists(W2_CKPT):
    w2_model = keras.models.load_model(W2_CKPT)
    # Week 2 model was trained with normalised [0,1] images
    def load_and_preprocess_w2(path, label):
        raw   = tf.io.read_file(path)
        image = tf.image.decode_jpeg(raw, channels=3)
        image = tf.image.resize(image, IMG_SIZE)
        image = tf.cast(image, tf.float32) / 255.0
        label = tf.one_hot(label, NUM_CLASSES)
        return image, label

    test_ds_w2 = (
        tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
        .map(load_and_preprocess_w2, num_parallel_calls=AUTOTUNE)
        .batch(32).prefetch(AUTOTUNE)
    )
    w2_loss, w2_acc, w2_top5 = w2_model.evaluate(test_ds_w2, verbose=0)
else:
    print(f"⚠️  Week 2 model not found at '{W2_CKPT}'. Using placeholder values.")
    w2_acc, w2_top5, w2_loss = 0.0, 0.0, 9.99

# Comparison bar chart
models_compared = ['Custom CNN\n(Week 2)', f'{BEST_BACKBONE}\n(Week 3 TL)']
accs   = [w2_acc,  tl_acc]
top5s  = [w2_top5, tl_top5]

x = np.arange(len(models_compared))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, [a*100 for a in accs],  width, label='Top-1 Accuracy (%)', color=['#4C72B0', '#DD8452'])
bars2 = ax.bar(x + width/2, [a*100 for a in top5s], width, label='Top-5 Accuracy (%)', color=['#4C72B0', '#DD8452'], alpha=0.5, hatch='//')

ax.axhline(90, color='red', linestyle='--', linewidth=1.5, label='90% Target')

for bar in bars1 + bars2:
    ax.annotate(
        f'{bar.get_height():.2f}%',
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 4), textcoords='offset points',
        ha='center', va='bottom', fontsize=9, fontweight='bold'
    )

ax.set_xticks(x)
ax.set_xticklabels(models_compared, fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_ylim(0, 105)
ax.set_title('Week 2 Custom CNN vs Week 3 Transfer Learning\n(Test Set Accuracy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")

improvement = (tl_acc - w2_acc) * 100
print(f"\n📊 Accuracy improvement over Custom CNN: +{improvement:.2f} percentage points")

 Save Week 3 Artefacts

In [ ]:
# ── 14.1  Save final model
FINAL_TL_MODEL_PATH = 'plant_disease_tl_final.keras'
best_tl_model.save(FINAL_TL_MODEL_PATH)
print(f"Final TL model saved: {FINAL_TL_MODEL_PATH}")

# ── 14.2  Save hyperparameter results and summary
summary = {
    'week': 3,
    'best_backbone'       : BEST_BACKBONE,
    'best_learning_rate'  : BEST_LR,
    'best_batch_size'     : BEST_BS,
    'phase1_best_val_acc' : p1_best_val_acc,
    'phase2_best_val_acc' : p2_best_val_acc,
    'test_accuracy'       : tl_acc,
    'test_top5_accuracy'  : tl_top5,
    'test_loss'           : tl_loss,
    'target_achieved'     : bool(tl_acc >= 0.90)
}
with open('week3_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("Summary saved: week3_summary.json")

# ── 14.3  List all saved files
print("\n📦 Week 3 Artefacts:")
week3_files = [
    'tl_phase1_best.keras',
    'tl_phase2_best.keras',
    'plant_disease_tl_final.keras',
    'tl_phase1_log.csv',
    'tl_phase2_log.csv',
    'hp_search_results.csv',
    'week3_summary.json',
    'hp_search_heatmap.png',
    'tl_training_curves.png',
    'tl_confusion_matrix.png',
    'model_comparison.png',
]
for fname in week3_files:
    if os.path.exists(fname):
        size = os.path.getsize(fname)
        print(f"   ✅ {fname:45s} {size/1024:.1f} KB")
    else:
        print(f"   ⏳ {fname} — will be created on run")